In [1]:
import pandas as pd
import requests
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

TMDB_API_KEY = "cc0093b5ad09a876190e097a1c2c8e65"
base_url = "https://api.themoviedb.org/3"

In [2]:
"""API Call to Retrieve Top 1000 Actors with English Movies

Currently not in use since alternate method to retrieve top actors was used
"""

def get_top_actors(limit = 1000):
    actors = []
    page = 1

    while len(actors) < limit:
        try:
            response = requests.get(f"{base_url}/person/popular", params={
                'api_key' : TMDB_API_KEY,
                'page' : page
            })
            response.raise_for_status()
            response = response.json()

        except requests.exceptions.RequestException as e:
            print(f"Error: {e}")
            break

        for person in response.get('results', []):
            is_english = any(item.get('original_language') == "en" for item in person.get('known_for', []))
            is_actor = person.get('known_for_department') == "Acting"

            if is_english and is_actor:
                actors.append({
                    'id' : person['id'],
                    'name' : person['name']
                })

            if len(actors) >= limit:
                break
        
        print(f"Collected: {len(actors)} actors (Page {page})", end='\r')
        page += 1
        time.sleep(0.01)

    return pd.DataFrame(actors)


#actor_list = get_top_actors()
#actor_list.to_csv("actor_list.csv", index=False)
#actor_list = pd.read_csv("actor_list.csv")

In [3]:
def get_tmdb_id(imdb_id):
    url = f"{base_url}/find/{imdb_id}"

    try:
        response = requests.get(url, params={
            'api_key' : TMDB_API_KEY,
            'external_source' : "imdb_id"
        })
        response.raise_for_status()
        response = response.json()

        results  = response.get('person_results', [])

        if results:
            return str(results[0]['id'])

    except requests.exceptions.RequestException as e:
        print(f"Error: {e}")
        return None


#actor_list = pd.read_csv("../data/revised-data/top_actors.csv")
#actor_list = actor_list.drop(columns=['Position', 'Created', 'Modified', 'Description', 'Known For', 'Birth Date'])
#actor_list['tmdb_id'] = actor_list['Const'].apply(get_tmdb_id)
#actor_list = actor_list.drop(columns=['Const']).rename(columns={'Name' : 'name'})
#actor_list.to_csv("../data/revised-data/actor_list.csv", index=False)
actor_list = pd.read_csv("../data/revised-data/actor_list.csv")

In [4]:
"""Single API Call to Retrieve List of Movies Actor is In"""

def get_movies(person_id):
    url = f"{base_url}/person/{person_id}/movie_credits"
    
    try:
        response = requests.get(url, params={
            'api_key' : TMDB_API_KEY
        })
        response.raise_for_status()
        response = response.json()

    except requests.exceptions.RequestException as e:
        print(f"Error: {e}")
        return None
    
    cast_list = pd.DataFrame(response['cast'])
    columns = ['id', 'original_language', 'original_title']
    cast_list = cast_list[columns].rename(columns={'id' : 'movie_id'})

    return cast_list

In [5]:
"""Executes get_movies() for List of Actors"""

def get_all_movies(actor_ids, max_workers = 10):
    full_list = []
    total = len(actor_ids)
    completed = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(get_movies, actor_id): actor_id for actor_id in actor_ids}

        for future in as_completed(futures):
            actor_id = futures[future]
            completed += 1
            
            try:
                result = future.result()
                if result is not None and not result.empty:
                    result['actor_id'] = actor_id
                    full_list.append(result)
                    print(f"[{completed}/{total}] ✓ Actor {actor_id} — {len(result)} movies found")
                else:
                    print(f"[{completed}/{total}] ✗ Actor {actor_id} — no results")

            except Exception as e:
                print(f"Actor ID: {actor_id} encountered an error: {e}")
    
    full_df = pd.concat(full_list, ignore_index=True)
    return full_df

#expanded_movie_list = get_all_movies(actor_list['tmdb_id'])
#movie_list = expanded_movie_list.groupby(['movie_id', 'original_language', 'original_title'])['actor_id']
#movie_list = movie_list.apply(lambda x: ",".join(x.astype(str))).reset_index()
#movie_list = movie_list.rename(columns={"actor_id" : "actors"})
#movie_list = movie_list[movie_list['actors'].str.contains(",")]
#movie_list = movie_list[movie_list['original_language'] == "en"]
#movie_list.to_csv("../data/revised-data/movie_list.csv", index=False)
movie_list = pd.read_csv("../data/revised-data/movie_list.csv")

In [6]:
"""Single API call to get remaining details needed"""

def get_details(movie_id):
    url = f"{base_url}/movie/{movie_id}"

    try:
        response = requests.get(url, params={
            'api_key' : TMDB_API_KEY
        })
        response.raise_for_status()
        response = response.json()

        results = {"id": movie_id,
                   "year": response.get("release_date", "")[:4],
                   "revenue": response.get("revenue")}
        
        return results

    except requests.exceptions.RequestException as e:
        print(f"Error: {e}")
        return {"id": None, "year": None, "revenue": None}

In [ ]:
def get_all_details(movie_ids, max_workers = 10):
    full_data = []
    completed = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(get_details, movie_id) for movie_id in movie_ids}

        for future in as_completed(futures):
            completed += 1

            try: 
                result = future.result()
                full_data.append(result)
            except Exception as e:
                print(f"Error: {e}")

            print(completed)
        
    df = pd.DataFrame(full_data, columns = ["id", "year", "revenue"])
    df = df.sort_values("id").reset_index(drop=True)

    return df

final_movie_details = get_all_details(movie_list["movie_id"], max_workers = 20)

In [36]:
inflation_data = pd.read_excel("../data/SeriesReport-20260302132218_6cc5fa.xlsx", header=11)
inflation_data['Annual'] = inflation_data['Annual'].fillna(inflation_data['Jan'])
inflation_data = inflation_data[['Year', 'Annual']]
cpi = pd.Series(inflation_data.set_index('Year')['Annual'])

def adjust_for_inflation(value, from_year, to_year = 2026, cpi = cpi):
    if from_year == to_year:
        return value
    
    years = range(from_year + 1, to_year + 1)
    multiplier = 1
    for year in years:
        rate = cpi[year] / 100
        multiplier = multiplier * (1 + rate)
        
    return round(value * multiplier, 2)

c:\Users\blkbd\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [79]:
final_movie_list = movie_list.merge(final_movie_details, left_on = 'movie_id', right_on = 'id', how = "left")
final_movie_list = final_movie_list.drop(columns=["original_language", "id"])
final_movie_list = final_movie_list[final_movie_list["year"] != ""]
final_movie_list = final_movie_list[final_movie_list["year"].isna() == False]
final_movie_list['year'] = final_movie_list['year'].astype(int)
final_movie_list = final_movie_list[final_movie_list['year'] >= 1958]
final_movie_list["adjusted_revenue"] = final_movie_list.apply(lambda row: adjust_for_inflation(row['revenue'], row['year']), axis=1)

In [85]:
final_movie_list.to_parquet("../data/revised-data/final_movies.parquet")
actor_list.to_parquet("../data/revised-data/actor_list.parquet")

In [87]:
final_movie_list

,movie_id,original_title,actors,year,revenue,adjusted_revenue
0,5,Four Rooms,"3131,3136,62,3141,3127,3129",1995,4257354.0,8.876780e+06
1,6,Judgment Night,"9777,2880",1993,12136938.0,2.679509e+07
2,11,Star Wars,"3,12248,4,2",1977,775398007.0,4.270401e+09
3,12,Finding Nemo,"5293,118,8783,19,13",2003,940335536.0,1.636219e+09
4,13,Forrest Gump,"31,35,6856,32,9640,33,21457",1994,677387716.0,1.454756e+09
...,...,...,...,...,...,...
16644,1641285,Reflections on 'the Little Dragon',"19429,51576",2001,0.0,0.000000e+00
16645,1641541,PaleyLive: It’s Always Sunny in Philadelphia 2...,"518,95101",2025,0.0,0.000000e+00
16646,1643508,Janet Jackson: Japanese Singles Collection - G...,"16866,938",2022,0.0,0.000000e+00
16650,1644700,Fats Waller Was Born Here,"70851,2047",2008,0.0,0.000000e+00
